<a href="https://colab.research.google.com/github/deeplearningmn/AIEng/blob/main/5.data/homework/%D0%BB%D0%B0%D0%B14(ganchimeg%2Czorigt).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
import urllib.request
from io import StringIO

# SparkSession үүсгэж байна
spark = SparkSession.builder.appName("ETL with PySpark").getOrCreate()

# Spark-ийн хувийн конфигурацийг шалгах
print(f"PySpark version: {spark.version}")

PySpark version: 3.5.1


In [3]:
import urllib.request

# URL болон хадгалах зам
url = "https://raw.githubusercontent.com/datasets/covid-19/master/data/countries-aggregated.csv"
file_path = "/content/countries-aggregated.csv"

# Файлыг татаж авах
urllib.request.urlretrieve(url, file_path)

('/content/countries-aggregated.csv',
 <http.client.HTTPMessage at 0x7dc87f231640>)

In [6]:
# Өгөгдөл унших (CSV форматаар)

# DataFrame-д хөрвүүлэх
df = spark.read.csv(file_path, header=True, inferSchema=True)



# Эхний 5 мөрийг харах
df.show(5)

# ДатаFrame-ийн бүтцийг шалгах
df.printSchema()

# Статистик мэдээллийг харах
df.describe().show()

+----------+-----------+---------+---------+------+
|      Date|    Country|Confirmed|Recovered|Deaths|
+----------+-----------+---------+---------+------+
|2020-01-22|Afghanistan|        0|        0|     0|
|2020-01-23|Afghanistan|        0|        0|     0|
|2020-01-24|Afghanistan|        0|        0|     0|
|2020-01-25|Afghanistan|        0|        0|     0|
|2020-01-26|Afghanistan|        0|        0|     0|
+----------+-----------+---------+---------+------+
only showing top 5 rows

root
 |-- Date: date (nullable = true)
 |-- Country: string (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Recovered: integer (nullable = true)
 |-- Deaths: integer (nullable = true)

+-------+-----------+-----------------+------------------+-----------------+
|summary|    Country|        Confirmed|         Recovered|           Deaths|
+-------+-----------+-----------------+------------------+-----------------+
|  count|     161568|           161568|            161568|           16156

In [23]:
# Шинэ багана нэмж байна: 'recoveredPercent' гэдэг баганад батлагдснаас эдгэсэн хувийг гаргах, Монгол улсаар шүүх
df_filtered = df.filter(df["Country"] == "Mongolia")
df = df.withColumn("recoveredPercent", df["Recovered"] / df["Confirmed"]*100 )

# Өгөгдлийн дараалал болон агрегаци
df_grouped = df_filtered.groupBy("Country").avg("recoveredPercent")

# df_grouped.show()
df_grouped.show()


+--------+---------------------+
| Country|avg(recoveredPrecent)|
+--------+---------------------+
|Mongolia|   44.420704955337214|
+--------+---------------------+



In [24]:
df.write.csv("/content/covid_aggregated.csv", header=True)
df.write.parquet("/content/covid_aggregated.parquet")